In [4]:
import os, time, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, precision_score, recall_score, accuracy_score)
import seaborn as sns

tf.random.set_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.family'] = 'DejaVu Sans'

IMG_SIZE   = 128
BATCH_SIZE = 16
DATA_DIR   = '/content/dummy_dataset' # Updated to use the dummy dataset
OUT_DIR    = '/mnt/user-data/outputs'

# Create the output directory if it does not exist
os.makedirs(OUT_DIR, exist_ok=True)

CLASSES    = ['lluvioso', 'nublado', 'soleado']   # alphabetical = keras default
N_CLASSES  = 3

print("=" * 65)
print("APRENDIZAJE PROFUNDO — CLASIFICACIÓN CLIMÁTICA")
print("=" * 65)
print(f"TensorFlow: {tf.__version__}")

# ============================================================
# PARTE A — DATASET & DATA AUGMENTATION
# ============================================================
print("\n[A] Cargando dataset con augmentation...")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_gen = train_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='training', seed=42, shuffle=True)

val_gen = val_datagen.flow_from_directory(
    DATA_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, class_mode='categorical',
    subset='validation', seed=42, shuffle=False)

print(f"  Train: {train_gen.samples} imágenes | Val: {val_gen.samples} imágenes")
print(f"  Clases: {train_gen.class_indices}")

# Guardar ejemplos del dataset
fig, axes = plt.subplots(3, 5, figsize=(13, 8))
fig.suptitle('Ejemplos del Dataset — Condiciones Climáticas', fontsize=13, fontweight='bold')
for ci, cls in enumerate(CLASSES):
    # Ensure the path exists before listing its contents
    cls_path = os.path.join(DATA_DIR, cls)
    if os.path.exists(cls_path):
        imgs = sorted(os.listdir(cls_path))[:5]
        for j, fn in enumerate(imgs):
            img = np.array(Image.open(os.path.join(cls_path, fn)))
            axes[ci, j].imshow(img)
            axes[ci, j].axis('off')
            if j == 0:
                axes[ci, j].set_ylabel(cls.capitalize(), fontsize=11, fontweight='bold',
                                        rotation=90, labelpad=5)
    else:
        # Handle case where class directory might not exist in dummy data if not fully generated
        print(f"Warning: Class directory {cls_path} not found for visualization.")
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_dataset_ejemplos.png', bbox_inches='tight')
plt.close()
print("  Ejemplos dataset guardados.")

# ============================================================
# PARTE B — CNN DESDE CERO
# ============================================================
# Arquitectura justificada:
# Bloque 1: 32 filtros 3×3 — detecta bordes/colores. BN estabiliza. Pool reduce a 64×64
# Bloque 2: 64 filtros 3×3 — detecta texturas (nubes, lluvia). BN+Dropout(0.25)
# Bloque 3: 128 filtros 3×3 — detecta patrones complejos (cielo, estructuras). Pool→16×16
# Dense 256 + Dropout(0.5) — combina features. Dropout evita sobreajuste en capas densas
# Salida Softmax 3 — probabilidad por clase

def build_cnn():
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # ── Bloque 1 ──
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)          # 64×64

    # ── Bloque 2 ──
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)          # 32×32
    x = layers.Dropout(0.25)(x)

    # ── Bloque 3 ──
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3,3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)          # 16×16
    x = layers.Dropout(0.25)(x)

    # ── Cabeza clasificadora ──
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(N_CLASSES, activation='softmax')(x)

    return Model(inputs, outputs, name='CNN_Propia')

model_cnn = build_cnn()
model_cnn.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'])
model_cnn.summary()
print(f"\n  Parámetros entrenables CNN: {model_cnn.count_params():,}")

callbacks_cnn = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-6, verbose=0),
]

print("\n[B] Entrenando CNN desde cero (máx 30 épocas, EarlyStopping patience=5)...")
t0 = time.time()
hist_cnn = model_cnn.fit(
    train_gen, epochs=30,
    validation_data=val_gen,
    callbacks=callbacks_cnn, verbose=1)
t_cnn = time.time() - t0
print(f"  Tiempo entrenamiento CNN: {t_cnn:.0f}s")

# Curvas CNN
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(hist_cnn.history['accuracy'],    color='#3498db', linewidth=2, label='Train')
axes[0].plot(hist_cnn.history['val_accuracy'],color='#e74c3c', linewidth=2, label='Val')
axes[0].set_title('CNN Propia — Accuracy', fontweight='bold')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(hist_cnn.history['loss'],    color='#3498db', linewidth=2, label='Train')
axes[1].plot(hist_cnn.history['val_loss'],color='#e74c3c', linewidth=2, label='Val')
axes[1].set_title('CNN Propia — Loss', fontweight='bold')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Curvas de Entrenamiento — CNN Desde Cero', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_cnn_curvas.png', bbox_inches='tight')
plt.close()

# Evaluación CNN
val_gen.reset()
y_pred_cnn = np.argmax(model_cnn.predict(val_gen, verbose=0), axis=1)
y_true     = val_gen.classes

acc_cnn = accuracy_score(y_true, y_pred_cnn)
f1_cnn  = f1_score(y_true, y_pred_cnn, average='macro')
print(f"\n  CNN — Accuracy: {acc_cnn:.4f} | F1 macro: {f1_cnn:.4f}")
print(classification_report(y_true, y_pred_cnn, target_names=CLASSES))

# Matriz de confusión CNN
fig, ax = plt.subplots(figsize=(6, 5))
cm_cnn = confusion_matrix(y_true, y_pred_cnn)
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASSES, yticklabels=CLASSES, cbar=False, linewidths=0.5)
ax.set_title(f'CNN Propia — Matriz de Confusión\nAcc={acc_cnn:.3f} | F1={f1_cnn:.3f}',
             fontweight='bold')
ax.set_xlabel('Predicción'); ax.set_ylabel('Real')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_cnn_confusion.png', bbox_inches='tight')
plt.close()

# ============================================================
# GRAD-CAM
# ============================================================
print("\n[B] Generando Grad-CAM para 3 imágenes...")

# Modelo con salida en última capa conv
last_conv = [l for l in model_cnn.layers if isinstance(l, layers.Conv2D)][-1].name
grad_model = Model(inputs=model_cnn.input,
                   outputs=[model_cnn.get_layer(last_conv).output,
                             model_cnn.output])

def get_gradcam(img_array, class_idx):
    """img_array: (1, H, W, 3) normalizado"""
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array, training=False)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(conv_out[0] * weights, axis=-1)
    cam = tf.maximum(cam, 0) / (tf.math.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()
    return cam, preds.numpy()[0]

# Seleccionar 3 imágenes de validación (una por clase)
val_gen.reset()
sample_imgs, sample_labels = next(val_gen)  # batch de 16
fig, axes = plt.subplots(3, 3, figsize=(12, 11))
fig.suptitle('Grad-CAM — Zonas Activadas por la CNN', fontsize=13, fontweight='bold')

for row, cls_idx in enumerate([0, 1, 2]):
    # Buscar una imagen de esta clase en el batch
    matches = np.where(np.argmax(sample_labels, axis=1) == cls_idx)[0]
    idx = matches[0] if len(matches) > 0 else row
    img_raw = sample_imgs[idx:idx+1]
    true_cls = np.argmax(sample_labels[idx])
    cam, preds = get_gradcam(img_raw, np.argmax(preds) if row > 0 else true_cls)
    pred_cls = np.argmax(preds)

    img_show = img_raw[0]

    axes[row, 0].imshow(img_show); axes[row, 0].axis('off')
    axes[row, 0].set_title(f'Original\nClase real: {CLASSES[true_cls]}', fontsize=9)

    axes[row, 1].imshow(cam, cmap='jet', vmin=0, vmax=1); axes[row, 1].axis('off')
    axes[row, 1].set_title('Mapa de Calor Grad-CAM', fontsize=9)

    overlay = img_show.copy()
    import matplotlib.cm as cm_mod
    heatmap_rgb = cm_mod.jet(cam)[:, :, :3]
    blended = 0.55 * overlay + 0.45 * heatmap_rgb
    axes[row, 2].imshow(np.clip(blended, 0, 1)); axes[row, 2].axis('off')
    correct = '✓' if pred_cls == true_cls else '✗'
    axes[row, 2].set_title(f'Superposición\nPred: {CLASSES[pred_cls]} {correct}', fontsize=9)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_gradcam.png', bbox_inches='tight')
plt.close()
print("  Grad-CAM guardado.")

# ============================================================
# PARTE C — TRANSFER LEARNING MobileNetV2
# ============================================================
print("\n[C] Transfer Learning — MobileNetV2")

base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False, weights='imagenet')

# ── Fase 1: Feature Extraction — congelar toda la base ──
base_model.trainable = False

def build_tl_head(base):
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(N_CLASSES, activation='softmax')(x)
    return Model(base.input, out, name='TL_Fase1')

model_tl1 = build_tl_head(base_model)
model_tl1.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy', metrics=['accuracy'])

trainable_params_f1 = sum([
    np.prod(v.shape) for v in model_tl1.trainable_variables])
print(f"\n  Fase 1 — Parámetros entrenables: {trainable_params_f1:,}")

print("  Entrenando Fase 1 (10 épocas)...")
t1 = time.time()
train_gen.reset(); val_gen.reset()
hist_tl1 = model_tl1.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=[EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy')],
    verbose=1)
t_tl1 = time.time() - t1
print(f"  Tiempo Fase 1: {t_tl1:.0f}s")

# ── Fase 2: Fine-Tuning — descongelar últimas 30 capas ──
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model_tl1.compile(
    optimizer=keras.optimizers.Adam(1e-5),   # LR muy bajo para no destruir pesos
    loss='categorical_crossentropy', metrics=['accuracy'])

trainable_params_f2 = sum([
    np.prod(v.shape) for v in model_tl1.trainable_variables])
print(f"\n  Fase 2 — Parámetros entrenables: {trainable_params_f2:,}")

print("  Entrenando Fase 2 / Fine-Tuning (10 épocas)...")
t2 = time.time()
train_gen.reset(); val_gen.reset()
hist_tl2 = model_tl1.fit(
    train_gen, epochs=10,
    validation_data=val_gen,
    callbacks=[EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy')],
    verbose=1)
t_tl2 = time.time() - t2
print(f"  Tiempo Fase 2: {t_tl2:.0f}s")

# Curvas TL
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Transfer Learning MobileNetV2 — Curvas de Entrenamiento',
             fontsize=12, fontweight='bold')
for i, (hist, title) in enumerate([(hist_tl1,'Fase 1 (Feature Extraction)'),
                                    (hist_tl2,'Fase 2 (Fine-Tuning)')]):
    axes[i,0].plot(hist.history['accuracy'],    '#3498db', linewidth=2, label='Train')
    axes[i,0].plot(hist.history['val_accuracy'],'#e74c3c', linewidth=2, label='Val')
    axes[i,0].set_title(f'{title} — Accuracy', fontweight='bold')
    axes[i,0].set_xlabel('Época'); axes[i,0].set_ylabel('Accuracy')
    axes[i,0].legend(); axes[i,0].grid(alpha=0.3)
    axes[i,1].plot(hist.history['loss'],    '#3498db', linewidth=2, label='Train')
    axes[i,1].plot(hist.history['val_loss'],'#e74c3c', linewidth=2, label='Val')
    axes[i,1].set_title(f'{title} — Loss', fontweight='bold')
    axes[i,1].set_xlabel('Época'); axes[i,1].set_ylabel('Loss')
    axes[i,1].legend(); axes[i,1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_tl_curvas.png', bbox_inches='tight')
plt.close()

# Evaluación TL
val_gen.reset()
y_pred_tl = np.argmax(model_tl1.predict(val_gen, verbose=0), axis=1)
acc_tl = accuracy_score(y_true, y_pred_tl)
f1_tl  = f1_score(y_true, y_pred_tl, average='macro')
print(f"\n  TL Final — Accuracy: {acc_tl:.4f} | F1 macro: {f1_tl:.4f}")
print(classification_report(y_true, y_pred_tl, target_names=CLASSES))

# Matrices de confusión TL
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
val_gen.reset()
# Fase 1 eval
base_model.trainable = False
model_tl1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
# Just use final model for both for visualization
for i, (cm_data, title) in enumerate([
    (confusion_matrix(y_true, y_pred_tl), f'TL Fase 2 (Fine-Tuning)\nAcc={acc_tl:.3f} F1={f1_tl:.3f}')
]):
    sns.heatmap(cm_data, annot=True, fmt='d', cmap='Greens', ax=axes[i],
                xticklabels=CLASSES, yticklabels=CLASSES, cbar=False, linewidths=0.5)
    axes[i].set_title(title, fontweight='bold')
    axes[i].set_xlabel('Predicción'); axes[i].set_ylabel('Real')
axes[1].axis('off')
plt.suptitle('Transfer Learning — Matriz de Confusión', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_tl_confusion.png', bbox_inches='tight')
plt.close()

# ============================================================
# TABLA COMPARATIVA
# ============================================================
print("\n[D] Tabla comparativa de modelos:")

params_cnn = model_cnn.count_params()
best_acc_tl1 = max(hist_tl1.history['val_accuracy'])
best_f1_tl1  = best_acc_tl1   # approximate

comp_data = {
    'Modelo':         ['CNN Propia', 'TL Fase 1\n(Feature Ext.)', 'TL Fase 2\n(Fine-Tuning)'],
    'Val Accuracy':   [f'{acc_cnn:.4f}', f'{best_acc_tl1:.4f}', f'{acc_tl:.4f}'],
    'F1 Macro':       [f'{f1_cnn:.4f}',  f'~{best_acc_tl1:.4f}', f'{f1_tl:.4f}'],
    'Tiempo (s)':     [f'{t_cnn:.0f}', f'{t_tl1:.0f}', f'{t_tl2:.0f}'],
    'Params entr.':   [f'{params_cnn:,}', f'{trainable_params_f1:,}', f'{trainable_params_f2:,}'],
    'Épocas':         [str(len(hist_cnn.history['accuracy'])), '10', '10'],
}

print(f"\n{'Modelo':<25} {'Acc':>8} {'F1':>8} {'Tiempo':>8} {'Params entr.':>14}")
print("-" * 68)
for i in range(3):
    print(f"{comp_data['Modelo'][i].replace(chr(10),' '):<25} "
          f"{comp_data['Val Accuracy'][i]:>8} "
          f"{comp_data['F1 Macro'][i]:>8} "
          f"{comp_data['Tiempo (s)'][i]:>8}s "
          f"{comp_data['Params entr.'][i]:>14}")

# Gráfico tabla comparativa
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
modelos = ['CNN Propia', 'TL Fase 1', 'TL Fase 2']
accs = [acc_cnn, best_acc_tl1, acc_tl]
f1s  = [f1_cnn,  best_acc_tl1, f1_tl]
params_list = [params_cnn, trainable_params_f1, trainable_params_f2]
colors3 = ['#3498db', '#e67e22', '#2ecc71']

axes[0].bar(modelos, accs, color=colors3, edgecolor='black', linewidth=0.7)
axes[0].set_title('Val Accuracy', fontweight='bold'); axes[0].set_ylim(0, 1.1)
axes[0].grid(axis='y', alpha=0.3)
for i,v in enumerate(accs): axes[0].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(modelos, f1s, color=colors3, edgecolor='black', linewidth=0.7)
axes[1].set_title('F1-Score Macro', fontweight='bold'); axes[1].set_ylim(0, 1.1)
axes[1].grid(axis='y', alpha=0.3)
for i,v in enumerate(f1s): axes[1].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[2].bar(modelos, [p/1000 for p in params_list], color=colors3, edgecolor='black', linewidth=0.7)
axes[2].set_title('Parámetros Entrenables (K)', fontweight='bold')
axes[2].set_ylabel('Miles de parámetros'); axes[2].grid(axis='y', alpha=0.3)
for i,v in enumerate(params_list): axes[2].text(i, v/1000*1.01, f'{v/1000:.0f}K', ha='center', fontsize=8)

plt.suptitle('Comparativa: CNN Propia vs Transfer Learning', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_comparativa.png', bbox_inches='tight')
plt.close()

# ============================================================
# PARTE D — AUDITORÍA Y ANÁLISIS DE ERRORES
# ============================================================
print("\n[D] Auditoría — imágenes mal clasificadas...")

val_gen.reset()
all_imgs, all_labels = [], []
for batch_x, batch_y in val_gen:
    all_imgs.append(batch_x)
    all_labels.append(batch_y)
    if len(all_imgs) * BATCH_SIZE >= val_gen.samples:
        break
all_imgs   = np.concatenate(all_imgs)[:val_gen.samples]
all_labels = np.concatenate(all_labels)[:val_gen.samples]
y_pred_all = np.argmax(model_cnn.predict(all_imgs, verbose=0), axis=1)
y_true_all = np.argmax(all_labels, axis=1)

wrong_idx = np.where(y_pred_all != y_true_all)[0]
print(f"  Imágenes mal clasificadas por CNN: {len(wrong_idx)}/{val_gen.samples}")

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
fig.suptitle('Imágenes Mal Clasificadas por la CNN Propia',
             fontsize=12, fontweight='bold')
for i, idx in enumerate(wrong_idx[:8]):
    ax = axes[i//4, i%4]
    ax.imshow(all_imgs[idx])
    ax.set_title(f'Real: {CLASSES[y_true_all[idx]]}\nPred: {CLASSES[y_pred_all[idx]]}',
                 fontsize=8, color='red')
    ax.axis('off')
for i in range(len(wrong_idx), 8):
    axes[i//4, i%4].axis('off')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/t4_errores.png', bbox_inches='tight')
plt.close()

print("\n✓ TAREA 4 COMPLETA — TODOS LOS GRÁFICOS GENERADOS")
print(f"  CNN Accuracy: {acc_cnn:.4f} | TL Final Accuracy: {acc_tl:.4f}")

APRENDIZAJE PROFUNDO — CLASIFICACIÓN CLIMÁTICA
TensorFlow: 2.20.0

[A] Cargando dataset con augmentation...
Found 12 images belonging to 3 classes.
Found 3 images belonging to 3 classes.
  Train: 12 imágenes | Val: 3 imágenes
  Clases: {'lluvioso': 0, 'nublado': 1, 'soleado': 2}
  Ejemplos dataset guardados.


Model: "CNN_Propia"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 128, 128, 32)   │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 64, 64, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 32, 32, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │             

 Total params: 323,619 (1.23 MB)

 Trainable params: 322,211 (1.23 MB)

 Non-trainable params: 1,408 (5.50 KB)


  Parámetros entrenables CNN: 323,619

[B] Entrenando CNN desde cero (máx 30 épocas, EarlyStopping patience=5)...
Epoch 1/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 9s 9s/step - accuracy: 0.2500 - loss: 2.0196 - val_accuracy: 0.3333 - val_loss: 1.0986 - learning_rate: 0.0010
Epoch 2/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.1667 - loss: 2.2362 - val_accuracy: 0.3333 - val_loss: 1.0986 - learning_rate: 0.0010
Epoch 3/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.3333 - loss: 1.8263 - val_accuracy: 0.3333 - val_loss: 1.0987 - learning_rate: 0.0010
Epoch 4/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0833 - loss: 2.1526 - val_accuracy: 0.3333 - val_loss: 1.0987 - learning_rate: 0.0010
Epoch 5/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.0833 - loss: 1.8891 - val_accuracy: 0.3333 - val_loss: 1.0987 - learning_rate: 5.0000e-04
Epoch 6/30
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5000 - loss: 1.7354 - val_accuracy: 0.3333 - val_loss: 1.0988 - learning_rate: 5.0000

### Creating a Dummy Dataset to Resolve `FileNotFoundError`

The previous error occurred because the `DATA_DIR` specified (`/home/claude/dataset`) was not found. To proceed, I will create a simple dummy dataset structure in `/content/dummy_dataset` with a few placeholder images for each class. Then, I will update the `DATA_DIR` variable in your main code cell to point to this newly created directory.

**Note:** This is a temporary solution to resolve the `FileNotFoundError`. For actual model training, you will need to replace this dummy data with your real image dataset, ensuring it is organized into subdirectories corresponding to your classes (e.g., `dummy_dataset/lluvioso`, `dummy_dataset/nublado`, `dummy_dataset/soleado`).

In [2]:
import os
from PIL import Image
import numpy as np

# Define the new dummy data directory
DUMMY_DATA_DIR = '/content/dummy_dataset'

# Create the main directory if it doesn't exist
os.makedirs(DUMMY_DATA_DIR, exist_ok=True)

# Create subdirectories for each class and add dummy images
CLASSES = ['lluvioso', 'nublado', 'soleado'] # Make sure these match your CLASSES variable
IMG_SIZE = 128 # Match your IMG_SIZE variable

print(f"Creating dummy dataset structure in: {DUMMY_DATA_DIR}")

for cls_name in CLASSES:
    class_dir = os.path.join(DUMMY_DATA_DIR, cls_name)
    os.makedirs(class_dir, exist_ok=True)
    print(f"  - Created directory: {class_dir}")

    # Create 5 dummy images for each class
    for i in range(5):
        # Create a blank white image using PIL
        dummy_image = Image.fromarray(np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8) + (i*50 % 255))
        img_path = os.path.join(class_dir, f'dummy_img_{i}.jpg')
        dummy_image.save(img_path)
        # print(f"    - Created dummy image: {img_path}")

print("Dummy dataset creation complete.")

# Update DATA_DIR to point to the newly created dummy dataset
# Note: This change needs to be reflected in the original cell to take effect for flow_from_directory.
# For this demonstration, we are setting DATA_DIR here, but in a real scenario,
# you would modify the DATA_DIR variable in the first cell directly or rerun the first cell.
# The next step will modify the original cell (z8y-9zR5LNCH) to use this path.

Creating dummy dataset structure in: /content/dummy_dataset
  - Created directory: /content/dummy_dataset/lluvioso
  - Created directory: /content/dummy_dataset/nublado
  - Created directory: /content/dummy_dataset/soleado
Dummy dataset creation complete.
